# Person1

In [1]:
# =========================================
# Coursework Version
# Late Fusion under LOVO
# Compare 3 classifiers:
#   1) Logistic Regression
#   2) Linear SVM
#   3) Random Forest
# =========================================
from google.colab import files
uploaded = files.upload()
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# =========================
# Config
# =========================
DATA_PATH = Path("BATCH1_96.xlsx")
OUTPUT_DIR = Path("coursework_latefusion_3classifiers_outputs_1")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
EPS = 1e-8

# =========================
# 1. Load data
# =========================
df = pd.read_excel(DATA_PATH)
print("Dataset shape:", df.shape)

# =========================
# 2. Column definitions
# =========================
openface_cols = [
    "AU01_r", "AU02_r", "AU04_r", "AU05_r", "AU06_r", "AU07_r", "AU09_r", "AU10_r",
    "AU12_r", "AU14_r", "AU15_r", "AU17_r", "AU20_r", "AU23_r", "AU25_r", "AU26_r", "AU45_r",
    "AU01_c", "AU02_c", "AU04_c", "AU05_c", "AU06_c", "AU07_c", "AU09_c", "AU10_c",
    "AU12_c", "AU14_c", "AU15_c", "AU17_c", "AU20_c", "AU23_c", "AU25_c", "AU26_c", "AU28_c", "AU45_c",
    "gaze_angle_x", "gaze_angle_y", "pose_Rx", "pose_Ry", "pose_Rz"
]

ppg_cols = [
    "bpm", "ibi", "sdnn", "sdsd", "rmssd", "pnn20", "pnn50",
    "hr_mad", "sd1", "sd2", "s", "sd1/sd2", "breathingrate"
]

required_cols = ["clip", "label"] + openface_cols + ppg_cols
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}")

# =========================
# 3. Prepare matrices
# =========================
X_face = df[openface_cols].copy()
X_ppg = df[ppg_cols].copy()
groups = df["clip"].values
y_raw = df["label"].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = [str(c) for c in label_encoder.classes_]

print("\nLabel distribution:")
print(pd.Series(y_raw).value_counts().sort_index())

print("\nClip distribution:")
print(df["clip"].value_counts().sort_index())

# =========================
# 4. Build classifier pipelines
# =========================
def build_lr_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=0.1,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ])

def build_svm_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=1.0,
            kernel="linear",
            class_weight="balanced",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])

def build_rf_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

classifier_builders = {
    "Logistic Regression": build_lr_pipeline,
    "Linear SVM": build_svm_pipeline,
    "Random Forest": build_rf_pipeline,
}

# =========================
# 5. Helper functions
# =========================
def evaluate_predictions(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

def save_confusion_matrix(y_true, y_pred, labels, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def inner_cv_score(X, y, groups, model_template):
    """
    Training-only LOVO score used for fusion weighting.
    Returns mean balanced accuracy.
    """
    logo_inner = LeaveOneGroupOut()
    scores = []

    if len(np.unique(groups)) < 2:
        return 0.5

    for tr_idx, val_idx in logo_inner.split(X, y, groups):
        X_tr = X.iloc[tr_idx]
        X_val = X.iloc[val_idx]
        y_tr = y[tr_idx]
        y_val = y[val_idx]

        if len(np.unique(y_tr)) < 2:
            continue

        model = clone(model_template)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(balanced_accuracy_score(y_val, pred))

    if len(scores) == 0:
        return 0.5

    return float(np.mean(scores))

def average_probabilities(prob_matrix):
    return prob_matrix.mean(axis=0)

def get_rf_feature_importance(fitted_pipeline, feature_names):
    clf = fitted_pipeline.named_steps["clf"]
    if not isinstance(clf, RandomForestClassifier):
        return None
    return pd.DataFrame({
        "feature": feature_names,
        "importance": clf.feature_importances_
    }).sort_values("importance", ascending=False)

# =========================
# 6. Run LOVO for each classifier
# =========================
summary_rows = []
clip_summary_rows = []
fold_rows = []
sample_prediction_rows = []
clip_prediction_rows = []

all_rf_face_importances = []
all_rf_ppg_importances = []

logo = LeaveOneGroupOut()

for clf_name, build_fn in classifier_builders.items():
    print("\n" + "=" * 70)
    print(f"Running classifier: {clf_name}")
    print("=" * 70)

    face_model_template = build_fn()
    ppg_model_template = build_fn()

    all_true = []
    all_pred = []

    clip_true_all = []
    clip_pred_all = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X_face, y, groups), start=1):
        test_clip = groups[test_idx][0]

        X_face_train = X_face.iloc[train_idx]
        X_face_test = X_face.iloc[test_idx]
        X_ppg_train = X_ppg.iloc[train_idx]
        X_ppg_test = X_ppg.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]
        train_groups = groups[train_idx]

        face_model = clone(face_model_template)
        ppg_model = clone(ppg_model_template)

        face_model.fit(X_face_train, y_train)
        ppg_model.fit(X_ppg_train, y_train)

        # Save RF feature importance only for RF model
        if clf_name == "Random Forest":
            face_imp = get_rf_feature_importance(face_model, openface_cols)
            ppg_imp = get_rf_feature_importance(ppg_model, ppg_cols)
            if face_imp is not None:
                face_imp["fold"] = fold
                face_imp["classifier"] = clf_name
                all_rf_face_importances.append(face_imp)
            if ppg_imp is not None:
                ppg_imp["fold"] = fold
                ppg_imp["classifier"] = clf_name
                all_rf_ppg_importances.append(ppg_imp)

        face_proba = face_model.predict_proba(X_face_test)
        ppg_proba = ppg_model.predict_proba(X_ppg_test)

        face_score = inner_cv_score(X_face_train, y_train, train_groups, face_model_template)
        ppg_score = inner_cv_score(X_ppg_train, y_train, train_groups, ppg_model_template)

        w_face = face_score / (face_score + ppg_score + EPS)
        w_ppg = ppg_score / (face_score + ppg_score + EPS)

        fused_proba = w_face * face_proba + w_ppg * ppg_proba
        pred_fused = np.argmax(fused_proba, axis=1)

        fold_metrics = evaluate_predictions(y_test, pred_fused)

        fold_rows.append({
            "Classifier": clf_name,
            "Fold": fold,
            "Heldout_Clip": test_clip,
            "True_Label": label_encoder.inverse_transform([y_test[0]])[0],
            "Face_Weight": w_face,
            "PPG_Weight": w_ppg,
            **fold_metrics
        })

        for local_i, global_i in enumerate(test_idx):
            sample_prediction_rows.append({
                "Classifier": clf_name,
                "Fold": fold,
                "Clip": test_clip,
                "Row_Index": int(global_i),
                "True_Label": label_encoder.inverse_transform([y_test[local_i]])[0],
                "Pred_Label": label_encoder.inverse_transform([pred_fused[local_i]])[0],
            })

        all_true.extend(y_test.tolist())
        all_pred.extend(pred_fused.tolist())

        # clip-level aggregation
        fused_clip_proba = average_probabilities(fused_proba)
        pred_clip = int(np.argmax(fused_clip_proba))
        true_clip = int(y_test[0])

        clip_true_all.append(true_clip)
        clip_pred_all.append(pred_clip)

        clip_prediction_rows.append({
            "Classifier": clf_name,
            "Clip": test_clip,
            "True_Label": label_encoder.inverse_transform([true_clip])[0],
            "Pred_Label": label_encoder.inverse_transform([pred_clip])[0],
            "Face_Weight": w_face,
            "PPG_Weight": w_ppg,
        })

        print(
            f"Fold {fold:02d} | Clip={test_clip} | "
            f"Acc={fold_metrics['Accuracy']:.3f} | "
            f"BalAcc={fold_metrics['Balanced_Accuracy']:.3f} | "
            f"MacroPrecision={fold_metrics['Macro_Precision']:.3f} | "
            f"MacroF1={fold_metrics['Macro_F1']:.3f} | "
            f"Weights(face={w_face:.3f}, ppg={w_ppg:.3f})"
        )

    # overall sample-level
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    overall_metrics = evaluate_predictions(all_true, all_pred)

    summary_rows.append({
        "Classifier": clf_name,
        **overall_metrics
    })

    # overall clip-level
    clip_true_all = np.array(clip_true_all)
    clip_pred_all = np.array(clip_pred_all)
    clip_metrics = evaluate_predictions(clip_true_all, clip_pred_all)

    clip_summary_rows.append({
        "Classifier": clf_name,
        **clip_metrics
    })

    # save confusion matrices
    safe_name = clf_name.lower().replace(" ", "_")
    save_confusion_matrix(
        all_true, all_pred, class_names,
        f"Sample-level Confusion Matrix - {clf_name}",
        OUTPUT_DIR / f"cm_sample_{safe_name}.png"
    )
    save_confusion_matrix(
        clip_true_all, clip_pred_all, class_names,
        f"Clip-level Confusion Matrix - {clf_name}",
        OUTPUT_DIR / f"cm_clip_{safe_name}.png"
    )

    # save classification reports
    with open(OUTPUT_DIR / f"classification_report_{safe_name}.txt", "w", encoding="utf-8") as f:
        f.write(f"===== SAMPLE-LEVEL REPORT: {clf_name} =====\n")
        f.write(classification_report(all_true, all_pred, target_names=class_names, digits=4, zero_division=0))
        f.write("\n\n")
        f.write(f"===== CLIP-LEVEL REPORT: {clf_name} =====\n")
        f.write(classification_report(clip_true_all, clip_pred_all, target_names=class_names, digits=4, zero_division=0))

# =========================
# 7. Build coursework tables
# =========================
summary_df = pd.DataFrame(summary_rows)[
    ["Classifier", "Accuracy", "Balanced_Accuracy", "Macro_Precision", "Weighted_Precision", "Macro_F1", "Weighted_F1"]
]
clip_summary_df = pd.DataFrame(clip_summary_rows)[
    ["Classifier", "Accuracy", "Balanced_Accuracy", "Macro_Precision", "Weighted_Precision", "Macro_F1", "Weighted_F1"]
]
fold_df = pd.DataFrame(fold_rows)
sample_pred_df = pd.DataFrame(sample_prediction_rows)
clip_pred_df = pd.DataFrame(clip_prediction_rows)

# mean ± std table from per-fold results
fold_grouped = fold_df.groupby("Classifier")[[
    "Accuracy", "Balanced_Accuracy", "Macro_Precision", "Weighted_Precision", "Macro_F1", "Weighted_F1"
]].agg(["mean", "std"])

def format_mean_std(grouped_df, metric):
    return [
        f"{m:.3f} ± {s:.3f}"
        for m, s in zip(grouped_df[(metric, "mean")], grouped_df[(metric, "std")])
    ]

coursework_table = pd.DataFrame({
    "Classifier": fold_grouped.index,
    "Accuracy (mean ± std)": format_mean_std(fold_grouped, "Accuracy"),
    "Balanced Accuracy (mean ± std)": format_mean_std(fold_grouped, "Balanced_Accuracy"),
    "Macro Precision (mean ± std)": format_mean_std(fold_grouped, "Macro_Precision"),
    "Weighted Precision (mean ± std)": format_mean_std(fold_grouped, "Weighted_Precision"),
    "Macro-F1 (mean ± std)": format_mean_std(fold_grouped, "Macro_F1"),
    "Weighted-F1 (mean ± std)": format_mean_std(fold_grouped, "Weighted_F1"),
}).reset_index(drop=True)

# =========================
# 8. Export CSV files
# =========================
summary_df.to_csv(OUTPUT_DIR / "sample_level_summary.csv", index=False)
clip_summary_df.to_csv(OUTPUT_DIR / "clip_level_summary.csv", index=False)
coursework_table.to_csv(OUTPUT_DIR / "coursework_results_table_mean_std.csv", index=False)
fold_df.to_csv(OUTPUT_DIR / "per_fold_results.csv", index=False)
sample_pred_df.to_csv(OUTPUT_DIR / "sample_level_predictions.csv", index=False)
clip_pred_df.to_csv(OUTPUT_DIR / "clip_level_predictions.csv", index=False)

# =========================
# 9. Save RF feature importance
# =========================
if len(all_rf_face_importances) > 0:
    rf_face_df = pd.concat(all_rf_face_importances, ignore_index=True)
    rf_face_mean = (
        rf_face_df.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
    )
    rf_face_df.to_csv(OUTPUT_DIR / "rf_openface_feature_importance_all_folds.csv", index=False)
    rf_face_mean.to_csv(OUTPUT_DIR / "rf_openface_feature_importance_mean.csv", index=False)

if len(all_rf_ppg_importances) > 0:
    rf_ppg_df = pd.concat(all_rf_ppg_importances, ignore_index=True)
    rf_ppg_mean = (
        rf_ppg_df.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
    )
    rf_ppg_df.to_csv(OUTPUT_DIR / "rf_ppg_feature_importance_all_folds.csv", index=False)
    rf_ppg_mean.to_csv(OUTPUT_DIR / "rf_ppg_feature_importance_mean.csv", index=False)

# =========================
# 10. Print final tables
# =========================
print("\n================ SAMPLE-LEVEL SUMMARY ================")
print(summary_df)

print("\n================ CLIP-LEVEL SUMMARY ================")
print(clip_summary_df)

print("\n================ COURSEWORK TABLE (mean ± std) ================")
print(coursework_table)

print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")

Saving BATCH1_96.xlsx to BATCH1_96.xlsx
Dataset shape: (96, 55)

Label distribution:
0    24
1    30
2    18
3    24
Name: count, dtype: int64

Clip distribution:
clip
1     6
2     6
3     6
4     6
5     6
6     6
7     6
8     6
9     6
10    6
11    6
12    6
13    6
14    6
15    6
16    6
Name: count, dtype: int64

Running classifier: Logistic Regression
Fold 01 | Clip=1 | Acc=0.500 | BalAcc=0.500 | MacroPrecision=0.500 | MacroF1=0.333 | Weights(face=0.448, ppg=0.552)
Fold 02 | Clip=2 | Acc=0.000 | BalAcc=0.000 | MacroPrecision=0.000 | MacroF1=0.000 | Weights(face=0.500, ppg=0.500)
Fold 03 | Clip=3 | Acc=0.000 | BalAcc=0.000 | MacroPrecision=0.000 | MacroF1=0.000 | Weights(face=0.500, ppg=0.500)
Fold 04 | Clip=4 | Acc=0.000 | BalAcc=0.000 | MacroPrecision=0.000 | MacroF1=0.000 | Weights(face=0.500, ppg=0.500)
Fold 05 | Clip=5 | Acc=0.333 | BalAcc=0.333 | MacroPrecision=0.333 | MacroF1=0.167 | Weights(face=0.645, ppg=0.355)
Fold 06 | Clip=6 | Acc=0.000 | BalAcc=0.000 | MacroPrecis

# Person 2

In [2]:
# =========================================
# Coursework Version
# Late Fusion under LOVO
# Compare 3 classifiers:
#   1) Logistic Regression
#   2) Linear SVM
#   3) Random Forest
# =========================================
from google.colab import files
uploaded = files.upload()
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# =========================
# Config
# =========================
DATA_PATH = Path("BATCH2_96.xlsx")
OUTPUT_DIR = Path("coursework_latefusion_3classifiers_outputs_2")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
EPS = 1e-8

# =========================
# 1. Load data
# =========================
df = pd.read_excel(DATA_PATH)
print("Dataset shape:", df.shape)

# =========================
# 2. Column definitions
# =========================
openface_cols = [
    "AU01_r", "AU02_r", "AU04_r", "AU05_r", "AU06_r", "AU07_r", "AU09_r", "AU10_r",
    "AU12_r", "AU14_r", "AU15_r", "AU17_r", "AU20_r", "AU23_r", "AU25_r", "AU26_r", "AU45_r",
    "AU01_c", "AU02_c", "AU04_c", "AU05_c", "AU06_c", "AU07_c", "AU09_c", "AU10_c",
    "AU12_c", "AU14_c", "AU15_c", "AU17_c", "AU20_c", "AU23_c", "AU25_c", "AU26_c", "AU28_c", "AU45_c",
    "gaze_angle_x", "gaze_angle_y", "pose_Rx", "pose_Ry", "pose_Rz"
]

ppg_cols = [
    "bpm", "ibi", "sdnn", "sdsd", "rmssd", "pnn20", "pnn50",
    "hr_mad", "sd1", "sd2", "s", "sd1/sd2", "breathingrate"
]

required_cols = ["clip", "label"] + openface_cols + ppg_cols
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}")

# =========================
# 3. Prepare matrices
# =========================
X_face = df[openface_cols].copy()
X_ppg = df[ppg_cols].copy()
groups = df["clip"].values
y_raw = df["label"].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = [str(c) for c in label_encoder.classes_]

print("\nLabel distribution:")
print(pd.Series(y_raw).value_counts().sort_index())

print("\nClip distribution:")
print(df["clip"].value_counts().sort_index())

# =========================
# 4. Build classifier pipelines
# =========================
def build_lr_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=0.1,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ])

def build_svm_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=1.0,
            kernel="linear",
            class_weight="balanced",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])

def build_rf_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

classifier_builders = {
    "Logistic Regression": build_lr_pipeline,
    "Linear SVM": build_svm_pipeline,
    "Random Forest": build_rf_pipeline,
}

# =========================
# 5. Helper functions
# =========================
def evaluate_predictions(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

def save_confusion_matrix(y_true, y_pred, labels, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def inner_cv_score(X, y, groups, model_template):
    """
    Training-only LOVO score used for fusion weighting.
    Returns mean balanced accuracy.
    """
    logo_inner = LeaveOneGroupOut()
    scores = []

    if len(np.unique(groups)) < 2:
        return 0.5

    for tr_idx, val_idx in logo_inner.split(X, y, groups):
        X_tr = X.iloc[tr_idx]
        X_val = X.iloc[val_idx]
        y_tr = y[tr_idx]
        y_val = y[val_idx]

        if len(np.unique(y_tr)) < 2:
            continue

        model = clone(model_template)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(balanced_accuracy_score(y_val, pred))

    if len(scores) == 0:
        return 0.5

    return float(np.mean(scores))

def average_probabilities(prob_matrix):
    return prob_matrix.mean(axis=0)

def get_rf_feature_importance(fitted_pipeline, feature_names):
    clf = fitted_pipeline.named_steps["clf"]
    if not isinstance(clf, RandomForestClassifier):
        return None
    return pd.DataFrame({
        "feature": feature_names,
        "importance": clf.feature_importances_
    }).sort_values("importance", ascending=False)

# =========================
# 6. Run LOVO for each classifier
# =========================
summary_rows = []
clip_summary_rows = []
fold_rows = []
sample_prediction_rows = []
clip_prediction_rows = []

all_rf_face_importances = []
all_rf_ppg_importances = []

logo = LeaveOneGroupOut()

for clf_name, build_fn in classifier_builders.items():
    print("\n" + "=" * 70)
    print(f"Running classifier: {clf_name}")
    print("=" * 70)

    face_model_template = build_fn()
    ppg_model_template = build_fn()

    all_true = []
    all_pred = []

    clip_true_all = []
    clip_pred_all = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X_face, y, groups), start=1):
        test_clip = groups[test_idx][0]

        X_face_train = X_face.iloc[train_idx]
        X_face_test = X_face.iloc[test_idx]
        X_ppg_train = X_ppg.iloc[train_idx]
        X_ppg_test = X_ppg.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]
        train_groups = groups[train_idx]

        face_model = clone(face_model_template)
        ppg_model = clone(ppg_model_template)

        face_model.fit(X_face_train, y_train)
        ppg_model.fit(X_ppg_train, y_train)

        # Save RF feature importance only for RF model
        if clf_name == "Random Forest":
            face_imp = get_rf_feature_importance(face_model, openface_cols)
            ppg_imp = get_rf_feature_importance(ppg_model, ppg_cols)
            if face_imp is not None:
                face_imp["fold"] = fold
                face_imp["classifier"] = clf_name
                all_rf_face_importances.append(face_imp)
            if ppg_imp is not None:
                ppg_imp["fold"] = fold
                ppg_imp["classifier"] = clf_name
                all_rf_ppg_importances.append(ppg_imp)

        face_proba = face_model.predict_proba(X_face_test)
        ppg_proba = ppg_model.predict_proba(X_ppg_test)

        face_score = inner_cv_score(X_face_train, y_train, train_groups, face_model_template)
        ppg_score = inner_cv_score(X_ppg_train, y_train, train_groups, ppg_model_template)

        w_face = face_score / (face_score + ppg_score + EPS)
        w_ppg = ppg_score / (face_score + ppg_score + EPS)

        fused_proba = w_face * face_proba + w_ppg * ppg_proba
        pred_fused = np.argmax(fused_proba, axis=1)

        fold_metrics = evaluate_predictions(y_test, pred_fused)

        fold_rows.append({
            "Classifier": clf_name,
            "Fold": fold,
            "Heldout_Clip": test_clip,
            "True_Label": label_encoder.inverse_transform([y_test[0]])[0],
            "Face_Weight": w_face,
            "PPG_Weight": w_ppg,
            **fold_metrics
        })

        for local_i, global_i in enumerate(test_idx):
            sample_prediction_rows.append({
                "Classifier": clf_name,
                "Fold": fold,
                "Clip": test_clip,
                "Row_Index": int(global_i),
                "True_Label": label_encoder.inverse_transform([y_test[local_i]])[0],
                "Pred_Label": label_encoder.inverse_transform([pred_fused[local_i]])[0],
            })

        all_true.extend(y_test.tolist())
        all_pred.extend(pred_fused.tolist())

        # clip-level aggregation
        fused_clip_proba = average_probabilities(fused_proba)
        pred_clip = int(np.argmax(fused_clip_proba))
        true_clip = int(y_test[0])

        clip_true_all.append(true_clip)
        clip_pred_all.append(pred_clip)

        clip_prediction_rows.append({
            "Classifier": clf_name,
            "Clip": test_clip,
            "True_Label": label_encoder.inverse_transform([true_clip])[0],
            "Pred_Label": label_encoder.inverse_transform([pred_clip])[0],
            "Face_Weight": w_face,
            "PPG_Weight": w_ppg,
        })

        print(
            f"Fold {fold:02d} | Clip={test_clip} | "
            f"Acc={fold_metrics['Accuracy']:.3f} | "
            f"BalAcc={fold_metrics['Balanced_Accuracy']:.3f} | "
            f"MacroPrecision={fold_metrics['Macro_Precision']:.3f} | "
            f"MacroF1={fold_metrics['Macro_F1']:.3f} | "
            f"Weights(face={w_face:.3f}, ppg={w_ppg:.3f})"
        )

    # overall sample-level
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    overall_metrics = evaluate_predictions(all_true, all_pred)

    summary_rows.append({
        "Classifier": clf_name,
        **overall_metrics
    })

    # overall clip-level
    clip_true_all = np.array(clip_true_all)
    clip_pred_all = np.array(clip_pred_all)
    clip_metrics = evaluate_predictions(clip_true_all, clip_pred_all)

    clip_summary_rows.append({
        "Classifier": clf_name,
        **clip_metrics
    })

    # save confusion matrices
    safe_name = clf_name.lower().replace(" ", "_")
    save_confusion_matrix(
        all_true, all_pred, class_names,
        f"Sample-level Confusion Matrix - {clf_name}",
        OUTPUT_DIR / f"cm_sample_{safe_name}.png"
    )
    save_confusion_matrix(
        clip_true_all, clip_pred_all, class_names,
        f"Clip-level Confusion Matrix - {clf_name}",
        OUTPUT_DIR / f"cm_clip_{safe_name}.png"
    )

    # save classification reports
    with open(OUTPUT_DIR / f"classification_report_{safe_name}.txt", "w", encoding="utf-8") as f:
        f.write(f"===== SAMPLE-LEVEL REPORT: {clf_name} =====\n")
        f.write(classification_report(all_true, all_pred, target_names=class_names, digits=4, zero_division=0))
        f.write("\n\n")
        f.write(f"===== CLIP-LEVEL REPORT: {clf_name} =====\n")
        f.write(classification_report(clip_true_all, clip_pred_all, target_names=class_names, digits=4, zero_division=0))

# =========================
# 7. Build coursework tables
# =========================
summary_df = pd.DataFrame(summary_rows)[
    ["Classifier", "Accuracy", "Balanced_Accuracy", "Macro_Precision", "Weighted_Precision", "Macro_F1", "Weighted_F1"]
]
clip_summary_df = pd.DataFrame(clip_summary_rows)[
    ["Classifier", "Accuracy", "Balanced_Accuracy", "Macro_Precision", "Weighted_Precision", "Macro_F1", "Weighted_F1"]
]
fold_df = pd.DataFrame(fold_rows)
sample_pred_df = pd.DataFrame(sample_prediction_rows)
clip_pred_df = pd.DataFrame(clip_prediction_rows)

# mean ± std table from per-fold results
fold_grouped = fold_df.groupby("Classifier")[[
    "Accuracy", "Balanced_Accuracy", "Macro_Precision", "Weighted_Precision", "Macro_F1", "Weighted_F1"
]].agg(["mean", "std"])

def format_mean_std(grouped_df, metric):
    return [
        f"{m:.3f} ± {s:.3f}"
        for m, s in zip(grouped_df[(metric, "mean")], grouped_df[(metric, "std")])
    ]

coursework_table = pd.DataFrame({
    "Classifier": fold_grouped.index,
    "Accuracy (mean ± std)": format_mean_std(fold_grouped, "Accuracy"),
    "Balanced Accuracy (mean ± std)": format_mean_std(fold_grouped, "Balanced_Accuracy"),
    "Macro Precision (mean ± std)": format_mean_std(fold_grouped, "Macro_Precision"),
    "Weighted Precision (mean ± std)": format_mean_std(fold_grouped, "Weighted_Precision"),
    "Macro-F1 (mean ± std)": format_mean_std(fold_grouped, "Macro_F1"),
    "Weighted-F1 (mean ± std)": format_mean_std(fold_grouped, "Weighted_F1"),
}).reset_index(drop=True)

# =========================
# 8. Export CSV files
# =========================
summary_df.to_csv(OUTPUT_DIR / "sample_level_summary.csv", index=False)
clip_summary_df.to_csv(OUTPUT_DIR / "clip_level_summary.csv", index=False)
coursework_table.to_csv(OUTPUT_DIR / "coursework_results_table_mean_std.csv", index=False)
fold_df.to_csv(OUTPUT_DIR / "per_fold_results.csv", index=False)
sample_pred_df.to_csv(OUTPUT_DIR / "sample_level_predictions.csv", index=False)
clip_pred_df.to_csv(OUTPUT_DIR / "clip_level_predictions.csv", index=False)

# =========================
# 9. Save RF feature importance
# =========================
if len(all_rf_face_importances) > 0:
    rf_face_df = pd.concat(all_rf_face_importances, ignore_index=True)
    rf_face_mean = (
        rf_face_df.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
    )
    rf_face_df.to_csv(OUTPUT_DIR / "rf_openface_feature_importance_all_folds.csv", index=False)
    rf_face_mean.to_csv(OUTPUT_DIR / "rf_openface_feature_importance_mean.csv", index=False)

if len(all_rf_ppg_importances) > 0:
    rf_ppg_df = pd.concat(all_rf_ppg_importances, ignore_index=True)
    rf_ppg_mean = (
        rf_ppg_df.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
    )
    rf_ppg_df.to_csv(OUTPUT_DIR / "rf_ppg_feature_importance_all_folds.csv", index=False)
    rf_ppg_mean.to_csv(OUTPUT_DIR / "rf_ppg_feature_importance_mean.csv", index=False)

# =========================
# 10. Print final tables
# =========================
print("\n================ SAMPLE-LEVEL SUMMARY ================")
print(summary_df)

print("\n================ CLIP-LEVEL SUMMARY ================")
print(clip_summary_df)

print("\n================ COURSEWORK TABLE (mean ± std) ================")
print(coursework_table)

print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")

Saving BATCH2_96.xlsx to BATCH2_96.xlsx
Dataset shape: (96, 55)

Label distribution:
0    30
1     6
2    36
3    24
Name: count, dtype: int64

Clip distribution:
clip
1     6
2     6
3     6
4     6
5     6
6     6
7     6
8     6
9     6
10    6
11    6
12    6
13    6
14    6
15    6
16    6
Name: count, dtype: int64

Running classifier: Logistic Regression
Fold 01 | Clip=1 | Acc=1.000 | BalAcc=1.000 | MacroPrecision=1.000 | MacroF1=1.000 | Weights(face=0.750, ppg=0.250)
Fold 02 | Clip=2 | Acc=0.000 | BalAcc=0.000 | MacroPrecision=0.000 | MacroF1=0.000 | Weights(face=0.772, ppg=0.228)
Fold 03 | Clip=3 | Acc=0.000 | BalAcc=0.000 | MacroPrecision=0.000 | MacroF1=0.000 | Weights(face=0.810, ppg=0.190)
Fold 04 | Clip=4 | Acc=1.000 | BalAcc=1.000 | MacroPrecision=1.000 | MacroF1=1.000 | Weights(face=0.727, ppg=0.273)
Fold 05 | Clip=5 | Acc=0.000 | BalAcc=0.000 | MacroPrecision=0.000 | MacroF1=0.000 | Weights(face=0.806, ppg=0.194)
Fold 06 | Clip=6 | Acc=1.000 | BalAcc=1.000 | MacroPrecis